# Adiabatic quantum linear solver

Both original 4 by 4 systems and all three three-qubit evolutions. Generate H0 and H1 from the model, retain the 7/50, 7/50 and 10/100 time schedules, and compare all eight amplitudes and direct-solve overlaps.

The native first-order formula preserves the Pauli mask order recorded in the retained program. The historical transpiler order and basis are unpinned, so the Notebook reports actual native amplitudes, overlap and static instruction depth. H1 uses the original matrix scale, while the direct solve uses its normalized inputs. The reported overlap includes population in both ancilla sectors.

## Load the two source models

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.quantum_linear_solvers.adiabatic_linear_solvers.solving_qlsp_with_aqc.overview.model import (
    SOURCE_MATRIX,
    SOURCE_RHS,
    decompose,
    spectrum,
)
from examples.synthesis.algorithms.quantum_linear_solvers.adiabatic_linear_solvers.solving_qlsp_with_aqc.overview.plots import (
    save_comparison,
    save_spectrum,
)
from examples.synthesis.algorithms.quantum_linear_solvers.adiabatic_linear_solvers.solving_qlsp_with_aqc.overview.workflow import (
    SCENARIOS,
    evolve,
    source_models,
)

print("Source matrix and vector:", SOURCE_MATRIX, SOURCE_RHS)
folder = Path("artifacts/development/adiabatic-linear-solver")


## Compute the source condition number

In [ ]:
original, perturbed = source_models()
print("Original condition number:", original.condition_number)


## Construct the two Hamiltonians

In [ ]:
print(original.h0)
print(original.h1)
print("Original term counts:", len(decompose(original.h0)), len(decompose(original.h1)))


## Inspect the linear interpolation

In [ ]:
points, eigenvalues = spectrum(original)


## Validate the spectrum grid

In [ ]:
assert points.shape == (100,) and eigenvalues.shape == (100, 8)


## Plot the entire spectrum

In [ ]:
display(Image(filename=str(save_spectrum(original, folder / "original-spectrum.png"))))


## Zoom into the zero eigenvalue gap

In [ ]:
display(
    Image(
        filename=str(save_spectrum(original, folder / "original-zoom.png", zoom=True))
    )
)


## Retain the original total time

In [ ]:
_, duration, steps = SCENARIOS[0]
print("Evolution time:", duration)


## Retain all the time slices

In [ ]:
print("Number of time slices:", steps)


## Inspect the ordered first-order sequence

In [ ]:
print("Apply H0 once and H1 in ten substeps at each time slice.")


## Build and run the source schedule

In [ ]:
first = evolve(original, duration, steps)


## Inspect the native program and full vector

In [ ]:
print("Original full state:", first.state.tolist())


## Read all basis words

In [ ]:
assert first.state.shape == (8,)


## Compare to the direct solve

In [ ]:
print("Original direct solution:", original.exact.tolist())
print("Original fidelity:", first.fidelity)
display(
    Image(
        filename=str(
            save_comparison(original, first, folder / "original-comparison.png")
        )
    )
)


## Read the native circuit depth

In [ ]:
print("Original native resources:", first.program.report["resources"])


## Construct the perturbed system

In [ ]:
print("Perturbed condition number:", perturbed.condition_number)
print(
    "Perturbed term counts:", len(decompose(perturbed.h0)), len(decompose(perturbed.h1))
)


## Plot the perturbed spectrum

In [ ]:
display(
    Image(
        filename=str(save_spectrum(perturbed, folder / "perturbed-zoom.png", zoom=True))
    )
)


## Use the same schedule on the perturbed system

In [ ]:
_, duration, steps = SCENARIOS[1]
second = evolve(perturbed, duration, steps)
print("Perturbed full state:", second.state.tolist())


## Read the direct comparison under perturbation

In [ ]:
print("Perturbed direct solution:", perturbed.exact.tolist())
print("Perturbed fidelity:", second.fidelity)
display(
    Image(
        filename=str(
            save_comparison(perturbed, second, folder / "perturbed-comparison.png")
        )
    )
)


## Read the perturbed circuit depth

In [ ]:
print("Perturbed native resources:", second.program.report["resources"])


## Double the time grid and change the total time

In [ ]:
_, duration, steps = SCENARIOS[2]
print("Refined time and slices:", duration, steps)


## Run the full refined evolution

In [ ]:
refined = evolve(perturbed, duration, steps)
print("Refined full state:", refined.state.tolist())


## Compare the refined run

In [ ]:
print("Refined fidelity:", refined.fidelity)
display(
    Image(
        filename=str(
            save_comparison(perturbed, refined, folder / "refined-comparison.png")
        )
    )
)


## Inspect the final native depth

In [ ]:
print("Refined native resources:", refined.program.report["resources"])
